# AtlasBridge — serve N-ATLaS on Kaggle

Serves **NCAIR1/N-ATLaS** as an OpenAI-compatible API on a free Kaggle GPU and gives it a public URL for the AtlasBridge gateway.

**Before you run (one-time):**
1. On Hugging Face, open https://huggingface.co/NCAIR1/N-ATLaS and accept the access conditions.
2. Create a Hugging Face **read** token (Settings → Access Tokens).
3. In this Kaggle notebook: **Settings → Accelerator → GPU T4 x2** and **Internet → On** (needs a phone-verified Kaggle account).
4. **Add-ons → Secrets**: add `HF_TOKEN` (your token) and `UPSTREAM_KEY` (any long random string — the gateway will use it). Tick both for this notebook.

Then **Run All** and keep the tab open. The public URL prints in step 5.

_N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies._

## 1. Check GPUs and install server packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Kaggle's preinstalled PyTorch is kept as-is. Installing vLLM upgrades PyTorch and breaks it on Kaggle.
!pip install -q fastapi uvicorn accelerate

## 2. Load secrets

In [ ]:
import os, secrets
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
os.environ["HF_TOKEN"] = _s.get_secret("HF_TOKEN")
try:
    os.environ["UPSTREAM_KEY"] = _s.get_secret("UPSTREAM_KEY")
except Exception:
    os.environ["UPSTREAM_KEY"] = secrets.token_urlsafe(32)
    print("No UPSTREAM_KEY secret found; generated one for this session:", os.environ["UPSTREAM_KEY"])
print("Secrets loaded.")

## 3. Choose the server

T4s don't support bf16, so the model runs in fp16, split across both GPUs. First start downloads ~16 GB and takes 5–10 minutes.

In [ ]:
import subprocess, time, requests

USE_FALLBACK = True  # transformers server (reliable on Kaggle). vLLM path needs `pip install vllm` and a matching PyTorch.

def start_vllm():
    cmd = [
        "vllm", "serve", "NCAIR1/N-ATLaS",
        "--served-model-name", "n-atlas",
        "--dtype", "half",
        "--tensor-parallel-size", "2",
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.90",
        "--api-key", os.environ["UPSTREAM_KEY"],
        "--port", "8000",
    ]
    return subprocess.Popen(cmd, stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())

if not USE_FALLBACK:
    server = start_vllm()
    print("vLLM starting (pid", server.pid, ") — logs in server.log")

## 3b. transformers server (default)

Loads N-ATLaS in fp16 across both T4s and serves an OpenAI-compatible API on port 8000.

In [ ]:
%%writefile fallback_server.py
"""
AtlasBridge fallback server.

A minimal OpenAI-compatible server for N-ATLaS using Hugging Face transformers.
Use this only if vLLM fails to start on the GPU. It is slower (one request at a
time) but works on almost any CUDA GPU, including Kaggle's 2x T4.

Endpoints:
  GET  /health
  GET  /v1/models
  POST /v1/chat/completions   (supports "stream": true)

Env:
  HF_TOKEN       Hugging Face token with access to NCAIR1/N-ATLaS
  UPSTREAM_KEY   Bearer key callers must send (optional but recommended)
  PORT           default 8000
"""

import json
import os
import threading
import time
import uuid
from datetime import datetime

import torch
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = "NCAIR1/N-ATLaS"
SERVED_NAME = "n-atlas"
HF_TOKEN = os.environ.get("HF_TOKEN")
API_KEY = os.environ.get("UPSTREAM_KEY", "")

print("Loading tokenizer...", flush=True)
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
print("Loading model (fp16, split across available GPUs)...", flush=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()

EOS_IDS = [tok.eos_token_id]
_eot = tok.convert_tokens_to_ids("<|eot_id|>")
if isinstance(_eot, int) and _eot not in EOS_IDS:
    EOS_IDS.append(_eot)

gen_lock = threading.Lock()  # one generation at a time
app = FastAPI(title="AtlasBridge fallback (N-ATLaS)")


def check_auth(request: Request) -> None:
    if API_KEY and request.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid upstream key")


def prepare(body: dict):
    messages = body.get("messages")
    if not messages:
        raise HTTPException(status_code=400, detail="'messages' is required")
    text = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        date_string=datetime.now().strftime("%d %b %Y"),
    )
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda:0")
    temperature = float(body.get("temperature", 0.1))
    max_tokens = int(body.get("max_tokens") or 512)
    kwargs = dict(
        max_new_tokens=max_tokens,
        repetition_penalty=float(body.get("repetition_penalty", 1.12)),
        do_sample=temperature > 0,
        use_cache=True,
        eos_token_id=EOS_IDS,
        pad_token_id=tok.eos_token_id,
    )
    if temperature > 0:
        kwargs.update(temperature=temperature, top_p=float(body.get("top_p", 0.9)))
    return inputs, kwargs, max_tokens


@app.get("/health")
def health():
    return {"status": "ok", "model": SERVED_NAME}


@app.get("/v1/models")
def models(request: Request):
    check_auth(request)
    return {"object": "list", "data": [{"id": SERVED_NAME, "object": "model", "owned_by": "awarri"}]}


@app.post("/v1/chat/completions")
async def chat(request: Request):
    check_auth(request)
    body = await request.json()
    inputs, kwargs, max_tokens = prepare(body)
    prompt_tokens = int(inputs["input_ids"].shape[1])
    cid = f"chatcmpl-{uuid.uuid4().hex[:24]}"
    created = int(time.time())

    if body.get("stream"):
        streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)

        def run():
            with gen_lock, torch.inference_mode():
                model.generate(**inputs, **kwargs, streamer=streamer)

        threading.Thread(target=run, daemon=True).start()

        def chunk(delta, finish=None):
            payload = {
                "id": cid,
                "object": "chat.completion.chunk",
                "created": created,
                "model": SERVED_NAME,
                "choices": [{"index": 0, "delta": delta, "finish_reason": finish}],
            }
            return f"data: {json.dumps(payload, ensure_ascii=False)}\n\n"

        def events():
            yield chunk({"role": "assistant"})
            for piece in streamer:
                if piece:
                    yield chunk({"content": piece})
            yield chunk({}, "stop")
            yield "data: [DONE]\n\n"

        return StreamingResponse(events(), media_type="text/event-stream")

    with gen_lock, torch.inference_mode():
        out = model.generate(**inputs, **kwargs)
    new_tokens = out[0][prompt_tokens:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    completion_tokens = int(new_tokens.shape[0])
    return {
        "id": cid,
        "object": "chat.completion",
        "created": created,
        "model": SERVED_NAME,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "length" if completion_tokens >= max_tokens else "stop",
            }
        ],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        },
    }


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", "8000")))

In [ ]:
if USE_FALLBACK:
    try: server.kill()
    except Exception: pass
    server = subprocess.Popen(["python", "fallback_server.py"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    print("Fallback server starting (pid", server.pid, ")")

## 4. Wait until the model is ready

In [ ]:
deadline = time.time() + 25 * 60
while time.time() < deadline:
    if server.poll() is not None:
        print("Server exited. Last log lines:\n")
        print("".join(open("server.log").readlines()[-40:]))
        raise SystemExit("Server crashed — switch to the fallback (step 3b).")
    try:
        if requests.get("http://localhost:8000/health", timeout=3).status_code == 200:
            print("N-ATLaS is up ✅"); break
    except Exception:
        pass
    time.sleep(15)
    print("...still loading", time.strftime("%H:%M:%S"), "|", (open("server.log").readlines() or [""])[-1].strip()[:120])
else:
    raise SystemExit("Timed out waiting for the server. Check server.log.")

## 5. Public URL (Cloudflare quick tunnel)

In [ ]:
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m: PUBLIC_URL = m.group(0); break
    time.sleep(2)
print("UPSTREAM_URL =", PUBLIC_URL)
print("Set this as UPSTREAM_URL in the gateway (it changes every session).")

## 6. Smoke test through the public URL

In [ ]:
time.sleep(5)
SYSTEM = "You are a friendly, helpful assistant. Reply in the same language the user writes in."
tests = {
    "Hausa": "Menene amfanin ruwa ga jikin dan Adam?",
    "Igbo": "Kedu ihe bụ uru mmiri na-arụ n'ahụ mmadụ?",
    "Yoruba": "Kini anfaani omi fun ara eniyan?",
    "English": "In two sentences, why is clean water important?",
}
for lang, q in tests.items():
    t0 = time.time()
    r = requests.post(f"{PUBLIC_URL}/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['UPSTREAM_KEY']}"},
        json={"model": "n-atlas", "messages": [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
              "temperature": 0.1, "max_tokens": 200, "repetition_penalty": 1.12},
        timeout=300)
    r.raise_for_status()
    print(f"--- {lang} ({time.time()-t0:.1f}s)\n{r.json()['choices'][0]['message']['content']}\n")

## 7. Keep alive
Leave this running while you test or record. Stop it to end the session.

In [ ]:
while True:
    alive = server.poll() is None and tunnel.poll() is None
    print(time.strftime("%H:%M:%S"), "server+tunnel up" if alive else "⚠️ something stopped — re-run steps 3–5", "|", PUBLIC_URL)
    time.sleep(120)